# Sprint 2 Live — Authentication, rate limits and secrets

Use this notebook with LS05–LS08. Save one personal copy and continue in it across all four sessions. Each session teaches a concept, demonstrates it in a small API example, then gives you a changed condition to predict and test. No Campus project, business-case preparation or provider account is required for the core exercises. Names such as `fieldcare`, `dispatch` and `partner` are identifiers in the supplied example; `dispatch` and `partner` represent two different callers.

## Session map and learning outcomes

| Session | Official learning outcome | Notebook route |
|---|---|---|
| LS05 | Apply API key authentication and safe secret handling to the project service. | `ls05-source` → `code-05` → `code-06` → `ls05-startup`; independent request variation |
| LS06 | Apply per-key rate limiting to the project service and observe behavior under burst traffic. | `ls06-source` → `code-08` → `code-09` → `ls06-renewal`; independent changed-order experiment |
| LS07 | Walk through the most common ways secrets leak (logs, prompts, repos) and apply patterns to prevent each. | `live-ls07-workspace` → before check → log/prompt edits → Git check → fresh-marker attempt |
| LS08 | Demo the project service with API key auth, secret handling, and per-key rate limiting in place. | Explain the earlier notebook examples, predict a peer's changed condition, then test one independently |

The outcomes retain the course-plan wording. Here you develop and practise those concepts using the notebook scaffold. Use the notebook's table of contents or search for a named cell/section; the cell names in code comments match the session guides. Run only the current session's sequence, rather than **Run all**: some cells deliberately wait for your edits, predictions or a real time window.

**Your starting state:** setup creates a separate Live workspace. Temporary caller credentials stay in memory. A helper owns one loopback HTTP server process and stops only that process when you start another. `127.0.0.1` belongs to the Colab runtime, not your laptop; no public tunnel is needed. Reusable helpers handle workspace/process housekeeping while the authentication attachment, policy, requests and source edits remain visible.

**Your finished state:** keep your own predictions, observed outputs and short explanations in text cells in this notebook. Export the Live ZIP after a session and download it alongside your saved notebook. An output or a reference result is not a substitute for your explanation.


## 1. Set up once, or restore your own work

1. In a new personal copy, leave `LIVE_ARCHIVE` empty and run `setup`, then `code-03`. Setup clones current `main` and installs the locked dependencies. It prints the source revision, workspace path and model names; it makes no provider request.
2. Keep the same `LIVE_FOLDER` within a runtime. Setup preserves an existing marked Live folder, including edits, but regenerates temporary caller values and stops the previous owned server. Start the relevant service again after setup. **Do not rerun setup or a start cell during a rate-limit renewal experiment.**
3. To continue after a runtime reset, upload your own exported Live ZIP through Colab's Files panel (or use its path in local Jupyter). Set `LIVE_ARCHIVE`, choose a `LIVE_FOLDER` that does not yet exist, and fill both `RESTORED_MODELS` from the saved `evidence/live-models.json` record or your notebook notes. The ZIP contains source/evidence, not Python variables; review the restored source, then run the current session's cells. Clear `LIVE_ARCHIVE` after a successful restore before rerunning setup in that runtime.

**Check:** only caller labels, model names and paths are displayed. A refused existing/unmarked folder or incompatible ZIP protects your work: choose a new empty destination and your own Live ZIP; do not delete or overwrite another workspace. A Campus checkpoint is not a Live ZIP. On install/import failure, retain the error and source revision, rerun setup once, and ask the instructor if it repeats. Credentials and indexes are excluded from exports; the optional provider section explains their separate preparation.


In [ ]:
# Cell: setup
from pathlib import Path
import os, sys, subprocess, tempfile, json, secrets, time
if globals().get("service") is not None:
    service.stop()
service = None
REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path(tempfile.mkdtemp(prefix="module-b-live-source-")) / "repo"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
import httpx
previous_helper_path = globals().get("LIVE_HELPER_PATH")
import importlib
if previous_helper_path in sys.path:
    sys.path.remove(previous_helper_path)
LIVE_HELPER_PATH = str(REPO / "src")
sys.path.insert(0, LIVE_HELPER_PATH)
# Select the new checkout even when setup is rerun in the same runtime.
for module_name in tuple(sys.modules):
    if module_name == "module_b" or module_name.startswith("module_b."):
        del sys.modules[module_name]
importlib.invalidate_caches()
from module_b.live import prepare_live_workspace, export_live_workspace, restore_live_workspace
from module_b.runtime import ServiceProcess
# On a fresh runtime, upload your own Live ZIP using the Files panel.
LIVE_ARCHIVE = ""  # Example: "/content/sprint-2-live-123.zip"; keep empty for new practice.
LIVE_FOLDER = Path.cwd() / "fieldcare-sprint-2-live"
# When restoring, explicitly select the same approved models used in that work.
RESTORED_MODELS = {"OPENROUTER_MODEL": "", "OPENROUTER_MODEL_V2": ""}
from module_b.openrouter import CHAT_MODELS, DEFAULT_CHAT_MODEL
if LIVE_ARCHIVE:
    if any(model not in CHAT_MODELS for model in RESTORED_MODELS.values()):
        raise ValueError("Fill both RESTORED_MODELS with your saved approved model choices before restoring.")
    PROJECT = restore_live_workspace(LIVE_ARCHIVE, LIVE_FOLDER)
else:
    PROJECT = prepare_live_workspace(REPO, LIVE_FOLDER)
CALLERS = {"dispatch": "FIELDCARE_DISPATCH_KEY", "partner": "FIELDCARE_PARTNER_KEY"}
caller_keys = {name: secrets.token_urlsafe(32) for name in CALLERS}
previous_models = {name: globals().get("service_env", {}).get(name) for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}
service_env = {variable: caller_keys[name] for name, variable in CALLERS.items()}
service_env["FIELDCARE_WORK_DIR"] = str((PROJECT / "var").resolve())
for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2"):
    service_env[name] = (RESTORED_MODELS[name] if LIVE_ARCHIVE else
                         previous_models[name] or os.environ.get(name, DEFAULT_CHAT_MODEL))
    if service_env[name] not in CHAT_MODELS:
        raise ValueError(f"Select an approved {name}.")
print("Configured models:", {name: service_env[name] for name in RESTORED_MODELS})
retry_at = None
renewal_pid = None
supported = {"question": "Which filter and airflow checks are documented?", "equipment_id": "EQ-FC-1002"}
incomplete = {"question": "Which filter checks are documented?"}
print("Live workspace:", PROJECT)
print("Source main:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())
print("No provider or index is needed for health, validation and clarification.")

from module_b.live import start_live_service


In [ ]:
# Cell: code-03
print("Caller labels:", list(CALLERS))
print("Temporary values remain in memory; values are never displayed.")


## LS05 — Caller keys, provider keys and runtime values

A **caller key** is a value the client sends to this API; the guard checks whether that caller is recognised. A **provider key** belongs to the API's separate request to a model provider. Passing the first check does not require the API to call a model: this example can validate input or return a clarification first.

### Read the mechanism, then make a prediction

Run `ls05-source`. Read `CALLER_ENV` as a mapping from caller labels to configuration **names**, not secret values. In `live_auth.py`, router registration happens before the protected POST-path inventory; `POLICY = None` means no rate limit in this session. The displayed guard loads actual values when the service is constructed, checks the request header, then allows input validation and the handler to run.

Run `code-05` to start that authentication-only example. **Expected:** an owned loopback URL. Read each condition in `code-06` and write a status prediction before running it. Missing/wrong caller credentials should be rejected; a recognised caller can still send an invalid body or receive a clarification. The matrix displays statuses without displaying keys. Its OpenAPI inventory includes schema-visible POST routes; compare the source registrations too rather than treating documentation as a complete security audit.

### Compare names with loaded values

Run `ls05-startup`: it stops the demo, removes one required caller value from a child environment, attempts startup, then restores a fresh service. **Expected:** startup refusal followed by a `200` clarification after restoration. Explain why a variable name in source does not supply its private runtime value. The notebook passes temporary caller values into a child process; it does not read them from a source file.

**Check:** connect each HTTP observation to the guard → validation → handler order. A `200` clarification is not evidence of model generation. The notebook does not measure an adapter-call counter. If calls refuse connection, run `code-05` again and repeat the observation; if a recognised caller suddenly gets `401` after setup, the caller values changed and the service needs restarting.

### Independent application — 14 minutes

Add a text cell for your prediction and a code cell copied from one familiar request in `code-06`. Change one caller condition or body condition, keeping the taught route and deterministic inputs. Predict before execution, run it, and explain the stopping point from source. Ask a peer to change one condition in your explanation. Keep the prediction, actual status and revised explanation; no completed variation is supplied. Use labels instead of credential values when sharing.


In [ ]:
# Cell: ls05-source
print((PROJECT / "service/fieldcare/security_settings.py").read_text())
print((PROJECT / "service/fieldcare/live_auth.py").read_text())
import inspect
from module_b.security import load_callers, SecurityMiddleware
print(inspect.getsource(load_callers))
print(inspect.getsource(SecurityMiddleware.__call__))


In [ ]:
# Cell: code-05
service = start_live_service(PROJECT, "live_auth", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
# Cell: code-06
auth_observations = []
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    paths = [path for path, methods in client.get("/openapi.json").json()["paths"].items() if "post" in methods]
    for path in paths:
        for label, key, body in [
            ("missing", None, incomplete), ("wrong", "synthetic-invalid", incomplete),
            ("admitted invalid", caller_keys["dispatch"], {}),
            ("admitted clarification", caller_keys["dispatch"], incomplete),
            ("second caller", caller_keys["partner"], incomplete),
        ]:
            response = client.post(path, json=body, headers={"X-API-Key": key} if key else {})
            auth_observations.append({"path": path, "condition": label, "http_status": response.status_code,
                                      "body_status": response.json().get("status")})
            print(path, label, response.status_code, response.json().get("status"))
    for path in ["/health", "/docs", "/openapi.json"]:
        print("public GET", path, client.get(path).status_code)
(PROJECT / "evidence").mkdir(exist_ok=True)
(PROJECT / "evidence/ls05-auth.json").write_text(json.dumps(auth_observations, indent=2) + "\n")
print("Safe matrix saved; keep your prediction and explanation in a text cell.")


In [ ]:
# Cell: ls05-startup
service.stop()
missing_env = {**service_env, "FIELDCARE_PARTNER_KEY": None}
try:
    rejected_start = start_live_service(PROJECT, "live_auth", missing_env)
except RuntimeError:
    print("Startup refused with a required caller value absent.")
else:
    rejected_start.stop()
    raise AssertionError("Expected configuration refusal")
service = start_live_service(PROJECT, "live_auth", service_env)
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    response = client.post("/v1/diagnose", json=incomplete, headers={"X-API-Key": caller_keys["dispatch"]})
    print("Restored configuration", response.status_code, response.json()["status"])

### Optional provider comparison — separate from the core checks

Leave both switches below `False` for the taught access, validation, clarification, rate-limit and secret-fixture work. These paths make no provider calls. If you choose a real generated-answer comparison, stop the service, obtain the provider key privately via Colab Secrets or the hidden prompt, and set `PREPARE_PROVIDER = True` to prepare the synthetic document index. Embedding uses provider quota. Only after successful preparation set `RUN_PROVIDER = True`; the next cell restarts the authentication example and makes one supported request.

**Expected:** the preparation command completes before the provider request. Inspect the returned status, response mode and answer; a healthy process or an accepted caller alone proves neither provider access nor answer quality. If preparation or generation fails, retain the bounded error, leave this comparison pending and continue the provider-free exercises. Restore both switches to `False` after the attempt to prevent accidental paid reruns. Never paste a provider value into a cell or display the environment.


In [ ]:
# Cell: prepare-provider
PREPARE_PROVIDER = False
if PREPARE_PROVIDER:
    if service is not None:
        service.stop()
    from module_b.openrouter import require_openrouter_key
    require_openrouter_key(prompt=True)
    preparation_env = {**os.environ, **service_env,
                       "PYTHONPATH": str(PROJECT / "service") + os.pathsep + str(REPO / "src")}
    subprocess.run([sys.executable, "-m", "fieldcare.prepare_index"],
                   cwd=PROJECT, env=preparation_env, check=True)
else:
    print("Provider preparation pending; deterministic checks remain available.")

In [ ]:
# Cell: ls05-provider
RUN_PROVIDER = False
if RUN_PROVIDER:
    service = start_live_service(PROJECT, "live_auth", service_env, service)
    with httpx.Client(base_url=service.base_url, timeout=120) as client:
        response = client.post("/v1/diagnose", json=supported, headers={"X-API-Key": caller_keys["dispatch"]})
        print(response.status_code, response.json())
else:
    print("Provider comparison pending; the access matrix uses deterministic clarification.")

## LS06 — Identity, shared allowances and renewal

A **rate limit** decides whether an authenticated caller has an admission available. This example allows **two admitted attempts per caller in a 60-second window**, shared across its protected routes. The window starts with that caller's first admission; it is not aligned to a wall-clock minute. Caller A is `dispatch`; caller B is `partner`. Their labels identify separate allowances, not different tasks.

1. Run `ls06-source` and locate `LimitPolicy(allowance=2, window_seconds=60)` and its middleware attachment. Explain why admission happens before body validation: an admitted invalid body consumes an attempt too.
2. Run `code-08` **once** for a fresh guided experiment. It starts the rate-limited service and clears any earlier renewal observation. Predict the five results in `code-09`, then run that cell once. **Expected categories:** authentication refusal, admitted invalid input, admitted clarification, exhausted caller, and another caller with its own allowance. The printed statuses and `Retry-After` support those distinctions. `401` and `429` do not consume an additional admission; admitted `422` does.
3. Keep the same process alive. Run `ls06-renewal` before expiry to see the remaining wait; after that delay rerun only that cell. **Expected:** the same worker accepts caller A again after expiry. Neither starting a new process nor seeing caller B succeed establishes A's renewal.

**Check:** changing from `/v1/diagnose` to `/v2/diagnose` must not give caller A a separate allowance. This experiment sends malformed or missing-context bodies, so no model call is needed. If you rerun `code-09`, its extra requests alter the buckets: save that observation, run `code-08` once for a clearly labelled new experiment, then repeat the sequence and wait. A stopped/replaced worker requires a new experiment; the renewal cell refuses to label it same-process evidence.


In [ ]:
# Cell: ls06-source
import inspect
print((PROJECT / "service/fieldcare/live_limits.py").read_text())
from module_b.security import FixedWindowLimiter
print(inspect.getsource(FixedWindowLimiter.admit))


In [ ]:
# Cell: code-08
service = start_live_service(PROJECT, "live_limits", service_env, service)
print("Owned HTTP service:", service.base_url)
retry_at = None
renewal_pid = None


In [ ]:
# Cell: code-09
if service is None or not service.is_running or service.app != "fieldcare.live_limits:app":
    raise RuntimeError("Run code-08 to start the rate-limit example first.")
retry_at = None
renewal_pid = service.pid
sequence_started = time.monotonic()
limit_observations = []
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    sequence = [
        ("missing", None, "/v1/diagnose", incomplete),
        ("A invalid counts", "dispatch", "/v1/diagnose", {}),
        ("A second admission", "dispatch", "/v2/diagnose", incomplete),
        ("A exhausted", "dispatch", "/v1/diagnose", incomplete),
        ("B independent", "partner", "/v1/diagnose", incomplete),
    ]
    for label, caller, path, body in sequence:
        headers = {"X-API-Key": caller_keys[caller]} if caller else {}
        response = client.post(path, json=body, headers=headers)
        print(round(time.monotonic() - sequence_started, 3), label, response.status_code,
              "Retry-After:", response.headers.get("retry-after"))
        limit_observations.append({"elapsed_seconds": round(time.monotonic() - sequence_started, 3),
                                   "condition": label, "caller": caller, "path": path,
                                   "http_status": response.status_code, "retry_after": response.headers.get("retry-after"),
                                   "worker_pid": renewal_pid})
        if response.status_code == 429:
            retry_after = int(response.headers["retry-after"])
            retry_at = time.monotonic() + retry_after
print("Keep this process running. Return to the next cell after Retry-After seconds.")
(PROJECT / "evidence").mkdir(exist_ok=True)
(PROJECT / "evidence/ls06-timeline.json").write_text(json.dumps(limit_observations, indent=2) + "\n")


In [ ]:
# Cell: ls06-renewal
if retry_at is None or renewal_pid is None:
    raise RuntimeError("Run code-08 once, then code-09 until it observes 429, before checking renewal.")
if service is None or not service.is_running or service.pid != renewal_pid:
    raise RuntimeError("The original worker is no longer running. Start a new labelled experiment; a restart is not renewal.")
remaining = max(0, retry_at - time.monotonic())
if remaining > 0:
    print("Wait", round(remaining, 1), "more seconds, then rerun this cell; do not restart.")
else:
    with httpx.Client(base_url=service.base_url, timeout=15) as client:
        response = client.post("/v1/diagnose", json=incomplete, headers={"X-API-Key": caller_keys["dispatch"]})
        print("Same-process renewal", response.status_code, response.json())
        limit_observations.append({"elapsed_seconds": round(time.monotonic() - sequence_started, 3),
                                   "condition": "renewal check", "caller": "dispatch", "path": "/v1/diagnose",
                                   "http_status": response.status_code, "worker_pid": service.pid})
        (PROJECT / "evidence/ls06-timeline.json").write_text(json.dumps(limit_observations, indent=2) + "\n")


### Independent changed-order experiment — 14 minutes

Add a text cell with your sequence and predicted results, then duplicate and edit `code-09` in this notebook. Start caller B first, include an admitted invalid body and two paths, and demonstrate renewal. Choose the ordering yourself. Run `code-08` once to begin your new experiment, then keep that worker alive through exhaustion, the other caller's request and expiry. Record times, labels and statuses; leave key values out.

Explain which observation establishes caller isolation, which establishes renewal, and why changing routes cannot reset a caller's allowance. Compare the actual trace with your prediction and explain any mismatch. Keep your own earlier output/notes before a rerun overwrites the latest safe timeline file. A passing guided sequence is not the completed independent attempt.


## LS07 — Secrets can be copied into logs, prompts and Git

Keeping a value out of a source file is only one protection. A logger can serialize headers or exception text; a prompt builder can copy runtime configuration into an outbound payload; Git can retain a committed value after the working file is ignored. Prevention happens at each copying boundary.

This workshop uses only a visibly synthetic, nonfunctional marker. It sends no HTTP or provider request. Run `live-ls07-workspace` to make a disposable local fixture, then inspect the intentionally unsafe before-state. You will edit two policy files and apply a Git prevention, inspect actual local artifacts, then repeat with a fresh marker independently.

**Editing in Colab:** use the folder path printed by the fixture cell in the Files panel, refresh the panel if needed, double-click the named `.py` file, edit and save it before running its check cell. In local Jupyter, open that same absolute path in your file editor. Edit files under `FIXTURE`, not the downloaded reference files under `REPO`. Each check uses `runpy` to reload your saved policy, so no service restart is needed.

**Fixture reruns:** reusing `RUN_NAME` preserves existing files; it does not reset them to the unsafe starter. Use a new name containing letters, digits or underscores for a fresh fixture. Do not delete an earlier attempt to obtain a desired result.


In [ ]:
# Cell: live-ls07-workspace
from module_b.secret_workshop import prepare_secret_workshop

RUN_NAME = "attempt_1"  # Change this to create a separate fresh fixture.
MARKER = f"SYNTHETIC_{RUN_NAME.upper()}_NOT_A_CREDENTIAL"
FIXTURE = prepare_secret_workshop(
    PROJECT / "secret-fixtures" / RUN_NAME,
    marker=MARKER,
    repo_root=REPO,
)
checks = {}
print("Disposable local fixture:", FIXTURE)
print("Provider calls:", 0, "| HTTP requests:", 0)


### Observe the deliberately unsafe before-state

Run `live-ls07-before`. It calls the starter `log_policy.py` and `prompt_policy.py` with synthetic values, writes actual local files and displays them. **Expected in a fresh fixture:** the marker appears in both ordinary/error log output and the assembled prompt payload. The local exception is a Python input to the logger, not an HTTP or provider failure.

Identify the copied field that introduced each marker before changing the policy. A previously edited fixture may already exclude it; choose a fresh `RUN_NAME` if you need the initial unsafe comparison. The prompt payload is assembled locally and is never sent to a model.


In [ ]:
# Cell: live-ls07-before
import json
import runpy

event = {
    "request_id": "synthetic-local-attempt-01",
    "route": "/v1/diagnose",
    "outcome": "completed",
    "headers": {"X-API-Key": MARKER},
}
log_record = runpy.run_path(str(FIXTURE / "log_policy.py"))["log_record"]
ordinary = log_record(event)
failed = log_record(
    {**event, "request_id": "synthetic-local-attempt-02", "outcome": "failed"},
    ValueError("Synthetic local exception contains " + MARKER),
)
LOG_PATH = FIXTURE / "observed.log"
LOG_PATH.write_text(json.dumps([ordinary, failed], indent=2) + "\n")

build_prompt = runpy.run_path(str(FIXTURE / "prompt_policy.py"))["build_prompt"]
payload = build_prompt(
    "Which filter check should the technician make?",
    "Synthetic HX guidance: inspect the intake filter.",
    {"provider_key": MARKER},
)
PAYLOAD_PATH = FIXTURE / "outbound-payload.json"
PAYLOAD_PATH.write_text(json.dumps(payload, indent=2) + "\n")

log_text = LOG_PATH.read_text()
payload_text = PAYLOAD_PATH.read_text()
print("Marker in saved ordinary/error logs:", MARKER in log_text)
print("Marker in locally built prompt payload:", MARKER in payload_text)
print("Saved ordinary/error log:")
display(json.loads(log_text))
print("Locally assembled outbound payload:")
display(json.loads(payload_text))


### Prevent disclosure before log serialization

Read `REPO/examples/patterns/secret_workshop/log_safe.py` as the guided pattern. In the Files panel, adapt that field selection in `FIXTURE/log_policy.py` and save. Retain request ID, route, outcome and a bounded error code; exclude request headers and raw exception text. The aim is useful diagnosis without copying credential-bearing inputs.

Run `live-ls07-log` and inspect `observed.log`. **Expected:** the marker is absent from both ordinary and error records, while request ID, route, outcome and an error code still explain the event. If a check fails, inspect the saved file and your policy's exception branch; removing only the ordinary header path is incomplete. Do not simply suppress the whole log record.


In [ ]:
# Cell: live-ls07-log
log_record = runpy.run_path(str(FIXTURE / "log_policy.py"))["log_record"]
ordinary = log_record(event)
failed = log_record(
    {**event, "request_id": "synthetic-local-attempt-02", "outcome": "failed"},
    ValueError("Synthetic local exception contains " + MARKER),
)
LOG_PATH.write_text(json.dumps([ordinary, failed], indent=2) + "\n")
log_text = LOG_PATH.read_text()
checks["ordinary and error logs exclude the marker"] = MARKER not in log_text
checks["safe diagnostic fields remain"] = all(
    row.get("request_id") and row.get("route") == "/v1/diagnose"
    and row.get("outcome") in {"completed", "failed"}
    for row in json.loads(log_text)
)
print("Marker in saved logs:", MARKER in log_text)
print("Retained record fields:", sorted(ordinary))


### Keep runtime configuration outside the prompt

Read `REPO/examples/patterns/secret_workshop/prompt_safe.py`, adapt its input selection in `FIXTURE/prompt_policy.py`, and save. A prompt needs the question and useful task context. Provider credentials configure the client that sends it; they are not part of its text.

Run `live-ls07-prompt` and inspect `outbound-payload.json`. **Expected:** no marker, with the question and useful context retained. A clean answer would not prove a clean outbound payload, so check the assembled request here. If context disappears too, repair the allowed inputs and rerun this cell. No request is sent to a model.


In [ ]:
# Cell: live-ls07-prompt
build_prompt = runpy.run_path(str(FIXTURE / "prompt_policy.py"))["build_prompt"]
payload = build_prompt(
    "Which filter check should the technician make?",
    "Synthetic HX guidance: inspect the intake filter.",
    {"provider_key": MARKER},
)
PAYLOAD_PATH.write_text(json.dumps(payload, indent=2) + "\n")
payload_text = PAYLOAD_PATH.read_text()
checks["outbound prompt excludes the marker"] = MARKER not in payload_text
checks["question and useful context remain"] = (
    "Which filter check" in payload_text and "inspect the intake filter" in payload_text
)
print("Marker in outbound payload:", MARKER in payload_text)
print("Question and context retained:", checks["question and useful context remain"])


### Distinguish ignore rules, the current index and old commits

The disposable fixture has a synthetic `.env` in its initial commit and an untracked `local.key`. Read `live-ls07-repository` before running it: it adds both names to `.gitignore`, then `git rm --cached` removes `.env` from the current index while leaving the local file. The Git index is the set of files staged/tracked for the next commit; ignore rules alone do not remove an already tracked file.

**Expected after the cell:** both names ignored, `.env` untracked, and the marker still present in the initial commit. The cell applies ignore and untrack together; it does not observe an intermediate state with only ignore rules. Explain why an old exposed credential would also need invalidation/rotation: ignoring or untracking does not revoke it or erase history. Here every value is nonfunctional and only the marked fixture repository is changed.


In [ ]:
# Cell: live-ls07-repository
(FIXTURE / ".gitignore").write_text(".env\nlocal.key\n")
subprocess.run(
    ["git", "-C", str(FIXTURE), "rm", "--cached", "--ignore-unmatch", ".env"],
    check=True, capture_output=True, text=True,
)
tracked = subprocess.run(
    ["git", "-C", str(FIXTURE), "ls-files", "--error-unmatch", ".env"],
    capture_output=True, text=True,
).returncode == 0
ignored_env = subprocess.run(
    ["git", "-C", str(FIXTURE), "check-ignore", "-q", ".env"]
).returncode == 0
ignored_key = subprocess.run(
    ["git", "-C", str(FIXTURE), "check-ignore", "-q", "local.key"]
).returncode == 0
initial_env = subprocess.run(
    ["git", "-C", str(FIXTURE), "show", "HEAD:.env"],
    check=True, capture_output=True, text=True,
).stdout
checks.update({
    "new .env is ignored": ignored_env,
    "new local.key is ignored": ignored_key,
    ".env is no longer tracked": not tracked,
    "initial commit still contains the marker": MARKER in initial_env,
})
print(".env ignored / tracked:", ignored_env, tracked)
print("local.key ignored:", ignored_key)
print("Marker remains in initial commit:", MARKER in initial_env)


### Independent fresh-marker attempt — 14 minutes

Choose a new `RUN_NAME`, predict where its marker will appear, then repeat the before check, both source edits and the Git prevention in that fresh fixture. Reuse the taught principles, but write your own explanation of what each rule excludes and what useful information it retains. Keep the before/after observations and explain why the old commit is a separate issue. No completed policy is inserted into this learner notebook.

Run `live-ls07-record` after all three checks. **Expected:** every named prevention/retention check is present and passes; a `REVIEW` or missing check means that condition is unresolved. Inspect the actual artifact and repair one rule before rerunning. This summary concerns this local synthetic fixture, not a scan of a personal project.

The summary retains your two edited policy files and selected synthetic results under `evidence/`. Rerun it after final edits, export and download the ZIP, and save your notebook notes. The disposable Git history, `.env`, `local.key` and raw fixture directory are excluded. An export preserves an incomplete attempt too; it does not certify completion.


In [ ]:
# Cell: live-ls07-record
expected_checks = {
    "ordinary and error logs exclude the marker", "safe diagnostic fields remain",
    "outbound prompt excludes the marker", "question and useful context remain",
    "new .env is ignored", "new local.key is ignored", ".env is no longer tracked",
    "initial commit still contains the marker",
}
for missing in sorted(expected_checks - checks.keys()):
    print("REVIEW — check not run:", missing)
for name, passed in checks.items():
    print(("PASS" if passed else "REVIEW"), name)
print("Evidence kind: local synthetic fixture")
print("Provider calls:", 0, "| HTTP requests:", 0)

# Preserve only the two edited policies and selected synthetic results in the Live ZIP.
import shutil
POLICY_EVIDENCE = PROJECT / "evidence/ls07-policies" / RUN_NAME
POLICY_EVIDENCE.mkdir(parents=True, exist_ok=True)
for name in ("log_policy.py", "prompt_policy.py"):
    shutil.copy2(FIXTURE / name, POLICY_EVIDENCE / name)
(PROJECT / "evidence" / f"ls07-{RUN_NAME}.json").write_text(json.dumps({
    "evidence_kind": "local synthetic fixture", "run_name": RUN_NAME,
    "checks": checks, "logs": json.loads(LOG_PATH.read_text()),
    "outbound_payload": json.loads(PAYLOAD_PATH.read_text()),
    "provider_calls": 0, "http_requests": 0,
}, indent=2) + "\n")
print("Retained for export:", POLICY_EVIDENCE)


## LS08 — Connect the controls in Colab

Use your own LS05 request matrix, continuous LS06 timeline and LS07 synthetic checks. Three different questions guide this review: **is the caller recognised; is an admission available; where might a secret be copied?** Source explains the mechanism; an HTTP or fixture observation establishes only the condition you actually tested.

1. Prepare one familiar request trace and one secret-copy example. Explain their rules without relying on the example's business names.
2. In three 8-minute rounds, rotate the explaining learner. Each round gives four minutes to explain the notebook examples, two minutes for a peer to change a caller/body/route/time or secret destination, one minute to compare the prediction with source, and one minute to save the revised explanation. A different caller tests isolation; renewal needs elapsed time in the same worker.
3. For your **12-minute independent check**, choose one taught condition to change. Write a prediction, run the relevant copied request or fresh synthetic fixture, and explain a mismatch or why the prediction held. Keep any unexecuted check explicitly pending. No new application project is required.

After a fresh restore, `evidence/ls05-auth.json` and `evidence/ls06-timeline.json` retain safe observations; `evidence/ls07-<RUN_NAME>.json` and `evidence/ls07-policies/<RUN_NAME>/` retain the synthetic checks and edited policies. Those are dated earlier attempts, not results from the new runtime. Regenerated caller values and a new worker require a new rate-limit timeline. Keep predictions and explanations in notebook text cells.

### Save your work

Run `save-checkpoint` only after finishing the active experiment: it stops the owned service, records the nonsecret model choices and creates a new ZIP. Download the printed ZIP through the Files panel and save your notebook. **Expected:** a unique ZIP path plus model names, with no credential values. If you need another HTTP observation after export, restart the relevant service; do not treat that new process as continuation of an earlier renewal experiment.


In [ ]:
# Cell: save-checkpoint
if service is not None:
    service.stop()
(PROJECT / "evidence").mkdir(exist_ok=True)
(PROJECT / "evidence/live-models.json").write_text(json.dumps(
    {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}, indent=2) + "\n")
print("Save these configured model choices with your notebook:",
      {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")})
CHECKPOINT_OUT = PROJECT.parent / f"sprint-2-live-{time.time_ns()}.zip"
print("Download:", export_live_workspace(PROJECT, CHECKPOINT_OUT))

## Summary

You can now distinguish caller and provider credentials, explain runtime configuration and the authentication/validation order, test a caller's shared allowance and real-time renewal, and prevent three secret-copy paths while retaining useful information. Your saved notebook connects a prediction, an observed result and the rule that explains it.

Keep this Live copy and its ZIP for review. Sprint 3 Live uses a separate notebook and workspace to practise streaming, logging and evaluation concepts.

## Quick recovery reference

| Symptom | Check and recovery |
|---|---|
| `PROJECT` or `service` is undefined | Run setup, then the current session's source/start cells. |
| Connection refused after export or runtime reset | Start the relevant owned service and make a new observation. |
| Statuses differ on a repeated burst | Earlier requests changed the allowance. Record them; start one new labelled experiment, then keep that worker alive. |
| Renewal cell reports no matching worker | The original process stopped or changed. Restart the experiment, not just the final request. |
| Synthetic marker still appears | Save the correct fixture policy and inspect ordinary/error logs or the outbound payload before rerunning. |
| Before-state is already safe | The named fixture preserved edits. Choose a fresh `RUN_NAME`; retain earlier work. |
| Restore refuses the folder/ZIP | Use your own Live ZIP and a destination that does not exist; fill the recorded model choices. |
| Optional provider work is unavailable | Keep both switches `False`; complete the provider-free core and mark the comparison pending. |
